In [29]:
from functools import cache
import re

def solve(start, goal, options, limit):
    @cache
    def dp(current, depth):
        if current == goal:
            return depth
        if depth >= limit:
            return float("inf")
        return min(dp(current ^ o, depth + 1) for o in options)

    result = dp(start, 0)
    return None if result == float("inf") else result


def parse_line(line):
    goal_str_raw = re.search(r"^\[([.#]+)\]", line).group(1)
    goal_str = goal_str_raw.translate(str.maketrans(".#", "01"))
    goal = int(goal_str, 2)
    
    size = len(goal_str)
    opts = re.findall(r"\(([,\d]+)\)", line)
    options = tuple(sum(1<<(size - 1 - int(i)) for i in o.split(",")) for o in opts)
    
    return goal, options

with open("input.txt") as f:
    print(sum(solve(0, *parse_line(line), 20) for line in f))


477


In [30]:
from z3 import *

def solve(buttons, goal):
    solver = Optimize()

    x = [Int(f"x{i}") for i in range(len(buttons))]
    for i in x:
        solver.add(i >= 0)

    for i in range(len(goal)):
        solver.add(Sum([buttons[b][i] * x[b] for b in range(len(buttons))]) == goal[i])

    solver.minimize(Sum(x))
    if solver.check() == sat:
        return sum(solver.model()[x[i]].as_long() for i in range(len(buttons)))
    return None

def parse_line(line):
    size = len(re.search(r"^\[([.#]+)\]", line).group(1))
    btns = [[*map(int, b.split(","))] for b in re.findall(r"\(([,\d]+)\)", line)]
    buttons = [[int(i in b) for i in range(size)] for b in btns]

    voltages = tuple(map(int, re.search(r"\{([^}]*)\}", line).group(1).split(",")))
    return buttons, voltages

with open("input.txt") as f:
    print(sum(solve(*parse_line(line)) for line in f))


17970
